This notebook spends a little time trying to get a basic feel for what's in the NVCS source tables through the lens of how we'll have to deal with the data in code.

In [65]:
import os
import pandas as pd
from IPython.display import display

For now, I dumped the files here into the working code repo to deal with them locally. Eventually, we will want to put these into their own source code repo or somehow deal with the version control dynamic on the files. I output the file list here to see what all I'm dealing with.

In [66]:
os.listdir("./USNVC_DownloadMarch2017")

['UnitObsoleteParent.txt',
 'reference.txt',
 'unit.txt',
 'UnitXReference.txt',
 'UnitObsoleteName.txt',
 'UnitXEcoregionUsfs2007.txt',
 'UnitXSubnation.txt',
 'UnitPredecessor.txt',
 'UnitSubtype.txt',
 'd_usfs_ecoregion1994.txt',
 'NVCS_DB_Diagram.png',
 'UnitParentSubtype.txt',
 'unitDescription.txt',
 'd_dist_confidence.txt',
 'd_occurrence_status.txt',
 'UnitXSimilarUnit.txt',
 'USNVCdatabase_29July2017.pdf',
 'd_curr_presence_absence.txt',
 'UnitXEcoregionUsfs1994.txt',
 'd_subnation.txt',
 'NVCS DB Keys.pdf',
 'd_usfs_ecoregion_level.txt']

There might be more efficient ways of processing these data from text files, but I find Pandas pretty convenient to work with. It introduces a few issues of its own, but we are going to need to deal with the basic messiness of going from text dumps from an unknown backend database platform into what we can work with cleanly. These are Windows sourced files with CRLF line terminators, which seem to be the default for C engine Pandas. We might also need to handle detecting the encoding of the content a little better, but I found ISO-8859-1 to work, at least.

In [71]:
units = pd.read_csv("USNVC_DownloadMarch2017/unit.txt", sep='\t', encoding = "ISO-8859-1")
display (units)

,element_global_id,parent_id,d_classification_level_id,elementuid,classificationcode,databasecode,status,colloquialname,scientificname,formattedscientificname,translatedname,hierarchylevel,unitsort,usstatus
0,860217,NaN,101,2.860217,1,C01,Accepted,Forest & Woodland,Mesomorphic Tree Vegetation Class,Mesomorphic Tree Vegetation Class,Mesomorphic Tree Vegetation Class,Class,1,CT
1,860211,NaN,101,2.860211,2,C02,Accepted,Shrub & Herb Vegetation,Mesomorphic Shrub & Herb Vegetation Class,Mesomorphic Shrub & Herb Vegetation Class,Mesomorphic Shrub & Herb Vegetation Class,Class,2,CT
2,860216,NaN,101,2.860216,3,C03,Accepted,Desert & Semi-Desert,"Xeromorphic Woodland, Scrub & Herb Vegetation ...","Xeromorphic Woodland, Scrub & Herb Vegetation ...","Xeromorphic Woodland, Scrub & Herb Vegetation ...",Class,3,CT
3,860213,NaN,101,2.860213,4,C04,Accepted,"Polar & High Montane Scrub, Grassland & Barrens","Cryomorphic Scrub, Herb & Cryptogam Vegetation...","Cryomorphic Scrub, Herb &amp; Cryptogam Vegeta...","Cryomorphic Scrub, Herb & Cryptogam Vegetation...",Class,4,CT
4,860214,NaN,101,2.860214,5,C05,Accepted,Aquatic Vegetation,Hydromorphic Vegetation Class,Hydromorphic Vegetation Class,Hydromorphic Vegetation Class,Class,5,CT
5,860218,NaN,101,2.860218,6,C06,Accepted,Open Rock Vegetation,Cryptogam - Open Mesomorphic Vegetation Class,Cryptogam - Open Mesomorphic Vegetation Class,Cryptogam - Open Mesomorphic Vegetation Class,Class,6,CT
6,860232,860217.0,102,2.860232,1.A,S17,Accepted,Tropical Forest & Woodland,Tropical Forest & Woodland Subclass,Tropical Forest &amp; Woodland Subclass,Tropical Forest & Woodland Subclass,Subclass,1.A,CT
7,860227,860217.0,102,2.860227,1.B,S15,Accepted,Temperate & Boreal Forest & Woodland,Temperate & Boreal Forest & Woodland Subclass,Temperate &amp; Boreal Forest &amp; Woodland S...,Temperate & Boreal Forest & Woodland Subclass,Subclass,1.B,CT
8,860219,860211.0,102,2.860219,2.A,S01,Accepted,"Tropical Grassland, Savanna & Shrubland","Tropical Grassland, Savanna & Shrubland Subclass","Tropical Grassland, Savanna & Shrubland Subclass","Tropical Grassland, Savanna & Shrubland Subclass",Subclass,2.A,CT
9,860233,860211.0,102,2.860233,2.B,S18,Accepted,Temperate & Boreal Grassland & Shrubland,Temperate & Boreal Grassland & Shrubland Subclass,Temperate & Boreal Grassland & Shrubland Subclass,Temperate & Boreal Grassland & Shrubland Subclass,Subclass,2.B,CT


In looking at the data, we seem to have several attributes that might be usable unique identifiers - element_global_id, elementuid, and databasecode. There are a few ways to test whether these are unique in the whole dataset, but I found one of the easiest things to do was get the size (shape) of the whole dataframe and then check unique size of the attributes (columns) in question. From this, it looks like any one of these is unique in the dataset, and I'm really not sure what the attributes mean and what their significance is. From other experience with NatureServe's data systems, it seems like the "Element Global ID" might be the unique ID to reference, but it's not clear whether or not it is persistent over time. Looking at the obsolete records might give us some clues as to how they handle things. We may opt to use our own assigned persistent ID depending on how we want to ultimately handle versioning of the records over time or our own way of dealing with deprecation.

In [72]:
print (units.shape)
print (units.element_global_id.unique().size)
print (units.elementuid.unique().size)
print (units.databasecode.unique().size)


(8416, 14)
8416
8416
8416


In exploring the ID issue a little further, I was curious about element_global_id and its persistence. This attribute appears to be the main identifier in the two tables containing information about obsolete entities. So, now I want to know if those presumably obsolete IDs show up back in the "live" units tables (at this point, I assume that unit.txt and unitDescription.txt basically have the same core unique identifiers. This is maybe a dumb way to do this test, but I read in the element_global_id values from all three tables throw unique values into arrays and then compare the arrays to see if there are any duplicates with unit.txt, where I expect to only find unique (and perhaps persistent element_global_id values).

In [91]:
obsoleteUnits = pd.read_csv("USNVC_DownloadMarch2017/UnitObsoleteName.txt", sep='\t', encoding = "ISO-8859-1", usecols=[0])
obsoleteParents = pd.read_csv("USNVC_DownloadMarch2017/UnitObsoleteParent.txt", sep='\t', encoding = "ISO-8859-1", usecols=[0])
uniqueUnits = units.element_global_id.unique()
uniqueObsoleteUnits = obsoleteUnits.element_global_id.unique()
uniqueObsoleteParents = obsoleteParents.element_global_id.unique()
print (any((uniqueUnits == uniqueObsoleteUnits).all() for uniqueUnits in uniqueObsoleteUnits))
print (any((uniqueUnits == uniqueObsoleteParents).all() for uniqueUnits in uniqueObsoleteParents))
print (any((uniqueObsoleteUnits == uniqueUnits).all() for uniqueObsoleteUnits in uniqueUnits))
print (any((uniqueObsoleteParents == uniqueUnits).all() for uniqueObsoleteParents in uniqueUnits))

False
False
False
False


I'm concluding (for the moment) that we should use element_global_id as a unique and persistent opaque identifier for a given unit in this data system over time. We still need to decide if we will adopt this as the core identifier for our purposes or assign our own for absolute certainty since we don't really know what all rules apply in the NatureServe Biotics system but suspect some level of squishiness given the multiple identifiers that seem to be in their data now. The "elementuid" is just a "2." with the element_global_id - don't know what that means, so maybe we can ignore it unless someone tells us something useful.

Both databasecode and classificationcode (don't understand the significance of the attribute names) have some part to play in establishing display titles for the units in the application. Certain databasecode values also appear in the one posted proceedings document from the ESA panel, where they seem to be used as meaningful shorthand (kind of a PITA, but as long as its consistent...). It appears that classificationcode is prefixed on Class, Subclass, Formation, and Division names and databasecode is prefixed on Macrogroup, Group, Alliance, and Association units to produce working full titles.

A different set of rules seems to apply for the Cultural units where the current app seems to be conflicting in terms of what is shown at a unit document level and what shows up in search results. Argh! We'll see if we can get a consistent ruleset nailed down that relates the identifiers to titl

parent_id appears to be a reference to a parent element_global_id, establishing the hierarchy, though we need to be careful about number handling with Pandas.